# Análisis de Supervivencia: Curvas de Kaplan-Meier

En este cuaderno exploramos la dinámica temporal del egreso en Matemáticas y Física. A diferencia del EDA estático, el análisis de supervivencia nos permite tratar correctamente a los alumnos que aún no terminan (datos censurados).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter
from pathlib import Path

# Configuración
DATOS_DIR = Path('../../Datos/Datos_limpios_activos')
sns.set_theme(style="whitegrid")

In [ ]:
def load_survival_data(carrera):
    df = pd.read_excel(DATOS_DIR / f'{carrera}.xlsx')
    # Definir T (tiempo) y E (evento)
    # Si semestre_termino es NaN o 20, consideramos que el evento NO ha ocurrido (censura)
    df['T'] = df['semestre_termino'].fillna(20)
    df['E'] = (df['semestre_termino'] < 20).astype(int)
    df['Carrera'] = carrera
    return df

df_mat = load_survival_data('Matemáticas')
df_fis = load_survival_data('Física')
df_all = pd.concat([df_mat, df_fis])

In [ ]:
kmf = KaplanMeierFitter()

plt.figure(figsize=(10, 6))

for carrera in ['Matemáticas', 'Física']:
    data = df_all[df_all['Carrera'] == carrera]
    kmf.fit(data['T'], event_observed=data['E'], label=carrera)
    kmf.plot_survival_function()

plt.title('Curvas de Kaplan-Meier: Probabilidad de seguir sin titularse', fontsize=15)
plt.xlabel('Semestres Transcurridos')
plt.ylabel('Probabilidad de No-Titulación')
plt.axvline(x=8, color='blue', linestyle='--', alpha=0.5, label='Ideal Mate (S8)')
plt.axvline(x=9, color='red', linestyle='--', alpha=0.5, label='Ideal Física (S9)')
plt.legend()
plt.show()

## Mediana de Tiempo de Egreso
La mediana de supervivencia es el tiempo en el que el 50% de la población ha experimentado el evento (titulación).